In [1]:
# Day 26 - Multi-Agent Systems

!pip install -q langchain langchain-community langchain-huggingface

from langchain_huggingface import HuggingFacePipeline
from langchain_core.prompts import PromptTemplate

import torch
from transformers import pipeline

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 24.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 2.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [2]:
# Load LLM
pipe = pipeline(
    "text-generation",
    model="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device=0 if torch.cuda.is_available() else -1,
    max_new_tokens=300,
    temperature=0.7
)

llm = HuggingFacePipeline(pipeline=pipe)
print("✅ LLM Loaded!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

...

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'temperature', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


✅ LLM Loaded!


In [3]:
# Define Two Agents

def researcher_agent(query):
    prompt = f"""You are a Researcher agent. Your job is to gather useful information.

Query: {query}

Provide key facts and sources (even if simulated)."""
    return llm.invoke(prompt)

def writer_agent(research, topic):
    prompt = f"""You are a Writer agent. Turn the research into a clear, engaging answer.

Topic: {topic}

Research: {research}

Write a well-structured response:"""
    return llm.invoke(prompt)

In [4]:
# Simple Multi-Agent Workflow
def multi_agent_system(query):
    print(f"🔍 Researcher is working on: {query}")
    research = researcher_agent(query)
    print("📝 Researcher Output:\n", research[:400], "...\n")

    print(f"✍️ Writer is creating final answer...")
    final_answer = writer_agent(research, query)
    print("✅ Final Answer:\n", final_answer)
    return final_answer

# Test
multi_agent_system("What are the main tourist attractions in Ethiopia?")

[transformers] Both `max_new_tokens` (=300) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🔍 Researcher is working on: What are the main tourist attractions in Ethiopia?


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer LlamaTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
[transformers] Both `max_new_tokens` (=300) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


📝 Researcher Output:
 You are a Researcher agent. Your job is to gather useful information.

Query: What are the main tourist attractions in Ethiopia?

Provide key facts and sources (even if simulated).

Narrator: In Ethiopia, there are many places to explore. Here are some of the main tourist attractions:

1. Addis Ababa - the capital city is home to numerous famous landmarks, including the Blue Nile Falls, the Bahir  ...

✍️ Writer is creating final answer...
✅ Final Answer:
 You are a Writer agent. Turn the research into a clear, engaging answer.

Topic: What are the main tourist attractions in Ethiopia?

Research: You are a Researcher agent. Your job is to gather useful information.

Query: What are the main tourist attractions in Ethiopia?

Provide key facts and sources (even if simulated).

Narrator: In Ethiopia, there are many places to explore. Here are some of the main tourist attractions:

1. Addis Ababa - the capital city is home to numerous famous landmarks, including the B

'You are a Writer agent. Turn the research into a clear, engaging answer.\n\nTopic: What are the main tourist attractions in Ethiopia?\n\nResearch: You are a Researcher agent. Your job is to gather useful information.\n\nQuery: What are the main tourist attractions in Ethiopia?\n\nProvide key facts and sources (even if simulated).\n\nNarrator: In Ethiopia, there are many places to explore. Here are some of the main tourist attractions:\n\n1. Addis Ababa - the capital city is home to numerous famous landmarks, including the Blue Nile Falls, the Bahir Dar Monastery, and the Ethnographic Museum.\n\n2. Tigray - known for its beautiful landscapes and ancient cultures, including the Temples of Debre Tabor and Debre Damo.\n\n3. Harar - a UNESCO World Heritage site, known for its historic fortifications, mosques, and charming markets.\n\n4. Lalibela - a UNESCO World Heritage site, known for its ancient rock-hewn churches, including the Red Church, the White Church, and the Black Church.\n\n5. 